# Parse ERA5 Running Total: Daily Mean, Calendar Conversion, and Regridding

Parse raw ERA5 daily min/max temperature data into a daily mean field,
convert to a no-leap calendar, and regrid to the target ECMWF S51 hindcast grid for
downstream bias adjustment.

**Data:**
- **Raw ERA5 daily min/max temperature** — `26_era5_daily_min.nc` / `26_era5_daily_max.nc`
- **Regrid target grid** — `s51_hcm.nc` (S51 seasonal hindcast ensemble mean, Copernicus)
- **Output (parsed, regridded ERA5)** — `26_era5_daily_running.zarr`

In [8]:
import datetime
import os
import uuid

import xarray as xr
import xesmf as xe  # type: ignore[ty:unresolved-import]
from dotenv import load_dotenv

In [ ]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]
ERA5_RUNNING_BASE = f"{DATA_DIR}/raw/26_era5_daily"

OUT_ZARR = f"{DATA_DIR}/parsed/2609_era5_daily_running.zarr"
TARGET_REGRID_URI = f"{DATA_DIR}/s51_hcm.nc"
UID = str(uuid.uuid4())
START_TIME = datetime.datetime.now(datetime.UTC).isoformat()

In [ ]:
def open_regrid_target(uri: str) -> xr.Dataset:
    """Open/clean a dataset to use as a regridding target"""
    # Using the S51 seasonal monthly seasonal hindcast ensemble mean from copernicus as the target grid for our regrid...
    # Selecting so only have coords for latitude and longitude for regridding.
    target = xr.open_dataset(uri).isel(
        {"forecast_reference_time": 0, "forecastMonth": 0}, drop=True
    )
    return target


def avg_era5(base_file):
    """Average the daily min and max surface air temperature"""
    # Open daily min and max era5 datasets
    era5_min = xr.open_dataset(f"{base_file}_min.nc")
    era5_max = xr.open_dataset(f"{base_file}_max.nc")
    # Daily average
    era5['tas'] = (era5_min['tas'] + era5_max['tas']) / 2
    # Rename the time dimension (CDS standard is "valid_time")
    era5 = era5.rename({"valid_time": "time"})
    return era5


def standardize_latlon(ds: xr.Dataset) -> xr.Dataset:
    """Harmonize latitude and longitude coordinates, returning a copy of the input dataset

    Input data with longitude from 0 to 360 is transformed to go from -180 to
    180 in ascending order. The "latitude" and "longitude" coordinates are renamed
    to "lat" and "lon", respectively.
    """
    _ds = ds.copy()

    _ds["longitude"] = (_ds["longitude"] + 180) % 360 - 180
    _ds = _ds.sortby("longitude")
    _ds = _ds.rename({"latitude": "lat", "longitude": "lon"})

    return _ds

In [11]:
regrid_target = open_regrid_target(TARGET_REGRID_URI)

In [ ]:
era5 = avg_era5(ERA5_RUNNING_BASE)
era5 = era5.rename({"t2m": "tas"})

In [18]:
# Cannot have leap years in QDM bias adjustment so convert to a no-leapyear calendar.
era5 = era5.convert_calendar("noleap", dim="time")
# Regrid the era5 dataset to the target grid (ECMWF)
regridder = xe.Regridder(era5, regrid_target, method="bilinear", periodic=True)
era5_regrid = regridder(era5)
era5_regrid.attrs |= era5.attrs

# Transform lat/lon coords for later projection.
era5_regrid = standardize_latlon(era5_regrid)

In [19]:
# Metadata on units is required later in the workflow.
era5_regrid["tas"].attrs["units"] = "K"

# Add additional general metadata.
era5_regrid.attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "Parsed ERA5 climate fields",
}
era5_regrid["tas"].attrs |= {
    "poreallas_created_at": START_TIME,
    "poreallas_uid": UID,
    "poreallas_description": "Parsed ERA5 tas field",
}

# All of time needs to be in a single chunk for QDM bias adjustment.
# This generally gets you ~110 MiB chunks.
era5_regrid = era5_regrid.chunk({"time": -1, "lat": 30, "lon": 60})

era5_regrid.to_zarr(OUT_ZARR, consolidated=True)
print(f"Output written to {OUT_ZARR}")

Output written to /home/emily_zuetell/projects/poreallas/data/parsed/2609_era5_daily_running.zarr


/home/emily_zuetell/miniforge3/envs/esmf-env/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
